# Notebook 03 — Google Trends: Search Interest by Beauty Tier

This Notebook attempts to directly assess user search interest by referencing Google searches, to see which beauty tier commands the highest attention in recent years.

## Methodology

**Approach:**
Google Trends data was pulled via pytrends for 2022–2025 across 8 markets.
Search interest is reported as a relative index (0–100 per batch), normalised 
against the anchor term "skincare" to enable cross-batch and cross-market comparison.
Values above 100 indicate search interest exceeding the average "skincare" baseline 
in that market. Numbers are therefore relative to the anchor, not absolute volumes.

**Geographies:** US, GB, FR, DE, JP, SG, TH, MY
Japanese queries used localised keyword variants (カタカナ/漢字) to capture 
native search behaviour. All other markets queried in English.

**Aggregation:**
Each keyword is tagged to a tier. Per-tier search index = mean across all 
keywords in that tier, then mean across all geographies. Monthly resampling 
applied for time series smoothing.

**Limitations:**
- Pytrends accesses Google's unofficial API — results may vary between runs
- Anchor normalisation assumes "skincare" has stable relative search volume 
  across markets, which may not hold equally in all geographies
- Korean market excluded due to romanisation limitations and absence of 
  Korean companies in the basket
- China excluded — Google Trends data unavailable
- Elevated indices in Japan and France partially reflect domestic brand familiarity and should be interpreted alongside cultural search behaviour differences."
- Search key terms are not exhaustive and were chosen as representatives of each tier. 
---

## Brand → Tier Assignments

| Tier | Brands |
|------|--------|
| **Luxury** | Estée Lauder, Clinique, La Mer, Tom Ford Beauty, Jo Malone, Bobbi Brown, Dior Beauty, Guerlain, Givenchy Beauty, Benefit, Fresh |
| **Prestige** | Lancôme, YSL Beauty, Giorgio Armani Beauty, Shiseido, NARS, Drunk Elephant, Clé de Peau, Decorté, Sekkisei, Dermalogica |
| **Masstige** | CeraVe, La Roche-Posay, SkinCeuticals, Vichy, Curél |
| **Mass** | L'Oréal Paris, Maybelline, NYX, Garnier, Dove, Simple, Bioré, Kanebo |

*Note: Tier assignments reflect the revenue segmentation in Notebook 01, 
not brand self-positioning. Some brands (e.g. Dermalogica, Curél) are 
positioned at tier boundaries and assigned based on primary consumer 
price perception.*

In [ ]:
# ── Notebook 03: Google Trends Analysis ───────────────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import time
import sys
sys.path.append("../src")
from helpers import TIER_COLOURS, set_style, save_chart
from pytrends.request import TrendReq

set_style()

In [ ]:
# ── Keyword → Tier mapping (English) ──────────────────────
TIER_KEYWORDS = {
    "Luxury": [
        "Estée Lauder", "Clinique", "La Mer", "Tom Ford Beauty",
        "Jo Malone", "Bobbi Brown", "Dior Beauty", "Guerlain",
        "Givenchy Beauty", "Benefit", "Fresh"
    ],
    "Prestige": [
        "Lancôme", "YSL Beauty", "Giorgio Armani Beauty",
        "Shiseido", "NARS", "Drunk Elephant", "Clé de Peau",
        "Decorté", "Sekkisei", "Dermalogica"
    ],
    "Masstige": [
        "CeraVe", "La Roche-Posay", "SkinCeuticals", "Vichy", "Curél"
    ],
    "Mass": [
        "L'Oréal Paris", "Maybelline", "NYX", "Garnier",
        "Dove", "Simple", "Bioré", "Kanebo"
    ],
}

# ── Japanese localised keywords ────────────────────────────
TIER_KEYWORDS_JP = {
    "Luxury": [
        "エスティローダー", "クリニーク", "ラ・メール", "トム フォード ビューティ",
        "ジョー マローン", "ボビイ ブラウン", "ディオール ビューティ", "ゲラン",
        "ジバンシィ ビューティ", "ベネフィット", "フレッシュ"
    ],
    "Prestige": [
        "ランコム", "イヴ・サンローラン ビューティ", "ジョルジオ アルマーニ ビューティ",
        "資生堂", "NARS", "ドランク エレファント", "クレ・ド・ポー ボーテ",
        "コスメデコルテ", "雪肌精", "デルマロジカ"
    ],
    "Masstige": [
        "CeraVe", "ラ ロッシュ ポゼ", "スキンシューティカルズ", "ヴィシー", "キュレル"
    ],
    "Mass": [
        "ロレアル パリ", "メイベリン", "NYX", "ガルニエ",
        "ダヴ", "シンプル", "ビオレ", "カネボウ"
    ],
}

# ── Geographies ────────────────────────────────────────────
GEO_MARKETS = {
    "US": "United States",
    "GB": "United Kingdom",
    "FR": "France",
    "DE": "Germany",
    "JP": "Japan",
    "SG": "Singapore",
    "TH": "Thailand",
    "MY": "Malaysia",
}

ANCHOR = "skincare"   # neutral anchor for cross-batch normalisation

In [ ]:
# ── Pytrends helper ────────────────────────────────────────
def fetch_batch(pytrends, keywords, geo, timeframe):
    """
    Fetch a single batch of up to 4 keywords + anchor term.
    Returns dataframe with normalised values relative to anchor.
    """
    batch = [ANCHOR] + keywords[:4]
    pytrends.build_payload(batch, geo=geo, timeframe=timeframe)
    time.sleep(1.5)   # rate limit buffer
    df = pytrends.interest_over_time()
    if df.empty or ANCHOR not in df.columns:
        return None
    # Normalise each keyword relative to anchor
    anchor_mean = df[ANCHOR].replace(0, 1).mean()
    result = {}
    for kw in keywords[:4]:
        if kw in df.columns:
            result[kw] = (df[kw] / anchor_mean * 100).values
    return pd.DataFrame(result, index=df.index)


def fetch_tier(pytrends, keywords, geo, timeframe):
    """
    Fetch all keywords for a tier in batches of 4.
    Returns single dataframe with all keywords.
    """
    chunks = [keywords[i:i+4] for i in range(0, len(keywords), 4)]
    frames = []
    for chunk in chunks:
        df = fetch_batch(pytrends, chunk, geo, timeframe)
        if df is not None:
            frames.append(df)
        time.sleep(2)
    if not frames:
        return None
    return pd.concat(frames, axis=1)

In [ ]:
# ── Main data pull (cache-enabled) ────────────────────────
import os, pickle

TRENDS_CACHE = "../data/processed/trends_raw.pkl"
USE_CACHE = True  # ← Set False to force a fresh pull

if USE_CACHE and os.path.exists(TRENDS_CACHE):
    print("Loading from cache...")
    with open(TRENDS_CACHE, "rb") as f:
        all_results = pickle.load(f)
    print("Loaded ✓")

else:
    df_master = pd.read_csv("../data/processed/master_revenue.csv")
    start_year = df_master["Year"].min()
    end_year   = df_master["Year"].max()
    timeframe  = f"{start_year}-01-01 {end_year}-12-31"

    pytrends = TrendReq(hl="en-US", tz=0, timeout=(10, 30))

    all_results = {}

    for tier, keywords in TIER_KEYWORDS.items():
        all_results[tier] = {}
        for geo_code, geo_name in GEO_MARKETS.items():
            print(f"Fetching {tier} / {geo_name}...")

            kw_list = TIER_KEYWORDS_JP[tier] if geo_code == "JP" else TIER_KEYWORDS[tier]

            df = fetch_tier(pytrends, kw_list, geo=geo_code, timeframe=timeframe)
            if df is not None:
                all_results[tier][geo_code] = df
            time.sleep(3)

    with open(TRENDS_CACHE, "wb") as f:
        pickle.dump(all_results, f)
    print("Cache saved ✓")

print("Done.")

In [ ]:
# ── Cache raw results to avoid re-fetching ─────────────────
import pickle
with open("../data/processed/trends_raw.pkl", "wb") as f:
    pickle.dump(all_results, f)
print("Saved: trends_raw.pkl")

In [ ]:
# ── Aggregate: mean search index per tier per geo ──────────
tier_geo_avg = {}

for tier, geos in all_results.items():
    tier_geo_avg[tier] = {}
    for geo, df in geos.items():
        # Normalise index to date only (strip time component if present)
        df.index = pd.to_datetime(df.index).normalize()
        tier_geo_avg[tier][geo] = df.mean(axis=1)

# Aggregate across geos → global tier trend
tier_global = {}
for tier, geos in tier_geo_avg.items():
    if geos:
        combined = pd.concat(geos.values(), axis=1)
        combined.columns = list(geos.keys())
        print(f"{tier}: combined shape={combined.shape}, nulls={combined.isna().sum().sum()}")
        tier_global[tier] = combined.mean(axis=1)

# Resample to monthly
tier_global_monthly = {
    tier: series.resample("ME").mean()
    for tier, series in tier_global.items()
}

print("\nMonthly series lengths:")
for tier, s in tier_global_monthly.items():
    print(f"  {tier}: {len(s)} months")

# ── Build trends_summary DataFrame & save ─────────────────
trends_summary = (
    pd.DataFrame(tier_global_monthly)
    .reset_index()
    .rename(columns={"index": "Date"})
)
trends_summary.to_csv("../data/processed/trends_summary.csv", index=False)
print("Saved ✓")
print(trends_summary.head())

In [ ]:
# ── Chart 1: Global tier trend over time ───────────────────
TIER_ORDER = ["Luxury", "Prestige", "Masstige", "Mass"]

fig, ax = plt.subplots()

for tier in TIER_ORDER:
    if tier in tier_global_monthly:
        series = tier_global_monthly[tier]
        ax.plot(series.index, series.values,
                label=tier,
                color=TIER_COLOURS[tier],
                linewidth=2.5)

ax.set_title("Google Search Interest by Beauty Tier (2022–2025)")
ax.set_xlabel("Date")
ax.set_ylabel("Normalised Search Index (anchor-relative)")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
ax.legend(title="Tier")

save_chart(fig, "03_search_trend_global.png")
plt.show()

In [ ]:
# ── Chart 2: Tier search index by geography (bar, avg over period) ──
geo_tier_means = {}
for geo_code, geo_name in GEO_MARKETS.items():
    geo_tier_means[geo_name] = {}
    for tier in TIER_ORDER:
        if geo_code in tier_geo_avg.get(tier, {}):
            geo_tier_means[geo_name][tier] = tier_geo_avg[tier][geo_code].mean()

df_geo = pd.DataFrame(geo_tier_means).T   # rows = geos, cols = tiers
df_geo = df_geo[TIER_ORDER]               # enforce column order

fig, ax = plt.subplots()
df_geo.plot(kind="bar", ax=ax,
            color=[TIER_COLOURS[t] for t in TIER_ORDER],
            width=0.7)

ax.set_title("Average Search Interest by Tier and Market (2022–2025)")
ax.set_xlabel("Market")
ax.set_ylabel("Normalised Search Index")
ax.legend(title="Tier")
plt.xticks(rotation=45)

save_chart(fig, "03_search_by_market.png")
plt.show()

In [ ]:
# ── Print geo × tier averages as a table ──────────────────
geo_tier_df = pd.DataFrame(geo_tier_means).T
geo_tier_df = geo_tier_df[TIER_ORDER].round(2)
print(geo_tier_df.to_string())

In [ ]:
df_trends_mean = (
    pd.DataFrame(tier_global_monthly)
    .mean()
    .reset_index()
    .rename(columns={"index": "Tier", 0: "Trends_Index"})
)

In [ ]:
# ── Dynamic Key Observations ─────────────────────────────
geo_summary = {}
for geo, geo_name in GEO_MARKETS.items():
    row = {}
    for tier in TIER_ORDER:
        if tier in all_results and geo in all_results[tier]:
            row[tier] = round(all_results[tier][geo].mean(axis=1).mean(), 1)
        else:
            row[tier] = None
    geo_summary[geo_name] = row

df_geo = pd.DataFrame(geo_summary).T
top_tier_global  = df_trends_mean.set_index("Tier")["Trends_Index"].idxmax()
low_tier_global  = df_trends_mean.set_index("Tier")["Trends_Index"].idxmin()
masstige_index   = df_trends_mean.loc[df_trends_mean["Tier"] == "Masstige", "Trends_Index"].values[0]
prestige_index   = df_trends_mean.loc[df_trends_mean["Tier"] == "Prestige", "Trends_Index"].values[0]
mass_index       = df_trends_mean.loc[df_trends_mean["Tier"] == "Mass",     "Trends_Index"].values[0]
luxury_index     = df_trends_mean.loc[df_trends_mean["Tier"] == "Luxury",   "Trends_Index"].values[0]

obs = f"""
## Key Observations

**Global pattern:**
- {top_tier_global} and Luxury dominate search volume (index: Mass {mass_index:.0f}, Luxury {luxury_index:.0f})
- Prestige and Masstige consistently low globally (Prestige: {prestige_index:.0f}), (Masstige: {masstige_index:.0f})
- Low search volume ≠ low engagement — Masstige highest revenue CAGR in NB01
  despite modest search index ({masstige_index:.0f} vs Mass {mass_index:.0f})

**Contextualising for geography**
- Japan: only market where Prestige exceeds luxury and rivals Mass — reflects premiumisation culture
- France: Masstige elevated — La Roche-Posay/Vichy as pharmacy staples
- Germany: strong pharmacy distribution of CeraVe/LRP inflates Masstige signal
- SEA: mass-dominant; modest Masstige signal via J/K-beauty crossover

**Implication for hypothesis:**
- Search data contextualises rather than contradicts Masstige revenue growth
  -> Am presently located in Japan, corroborates personal experience 
- Recent spike in Masstige interest towards the end of 2025 worth following to see if trend continues
- Generally, interest in Mass and Lxuury remain dominant  
"""
print(obs)